# AlphaFold 2 GPU baseline: reproducible re-run (free Colab)

Regenerates the AF2 GPU (Google Colab NVIDIA Tesla T4) baseline with the benchmark script **as committed in this repository**, instead of a copy pasted into the notebook.

What differs from the original 2026-08-08 run. That notebook, with its outputs, is preserved in git history at commit `faeaa4b`; see `paper/data/canonical_results.md`, discrepancies 11 and 12.

- **Script:** `src/spike_tpu_forward_pass.py` is cloned from the repo at a recorded commit and run with command-line flags: `model_3`, 0 recycles, 118 residues (the toy sequence), float32.
- **Repeats:** 5 steady-state `predict()` calls in the same process. Every value is saved, plus their mean and sample standard deviation. The first, compiling call can only happen once per process.
- **Profiler:** the timed first call runs without `jax.profiler.trace`, whose finalisation inflated the original first-call time. An optional second run with the profiler on measures that overhead on the same runtime.
- **Provenance:** the repo and AlphaFold 2 commits, package versions and hardware details are saved next to the results.

**Before running:** `Runtime > Change runtime type > T4 GPU`. Expect roughly 5 minutes for step 5 on a T4 (the original run took 109 s for `init_params`, 98 s for the first call, of which 42 s was profiler finalisation, and 13 s per steady-state call), plus about 4 minutes for the optional step 6.

## 1. Confirm the runtime has a Tesla T4

In [ ]:
import subprocess

# The published GPU baseline is a Tesla T4. Set this to False only if you
# deliberately want a different GPU (the numbers will not be comparable).
REQUIRE_T4 = True

smi = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                     capture_output=True, text=True)
assert smi.returncode == 0, "No GPU visible: switch to Runtime > Change runtime type > T4 GPU."
gpu_name = smi.stdout.strip()
assert not REQUIRE_T4 or "T4" in gpu_name, f"Expected a Tesla T4, got {gpu_name!r}."
print(f"GPU: {gpu_name}")
!nvidia-smi

## 2. Record the hardware

The original run did not record the CPU model, core count or RAM (`paper/sections/methodology.md`, Section 8).

In [ ]:
import json, os, platform, subprocess

def run(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

hardware = {
    "platform": platform.platform(),
    "python": platform.python_version(),
    "cpu_model": run("lscpu | sed -n 's/^Model name:[[:space:]]*//p'"),
    "logical_cpus": os.cpu_count(),
    "mem_total": run("grep MemTotal /proc/meminfo"),
    "lscpu": run("lscpu"),
    "nvidia_smi": None,
}
hardware["nvidia_smi"] = run(
    "nvidia-smi --query-gpu=name,driver_version,memory.total,compute_cap --format=csv")
print(json.dumps({k: v for k, v in hardware.items() if k != "lscpu"}, indent=2))

## 3. Install dependencies

JAX is pinned to 0.10.2, the version the TPU Jobs in `configs/` install. The other packages are left unpinned, as in the original runs; their resolved versions are recorded in step 7. Colab's preinstalled TensorFlow is used for AlphaFold's feature pipeline. `tensorflow-cpu` is deliberately not installed: next to Colab's GPU build it breaks the native extensions.

In [ ]:
!pip install -q "jax[cuda12]==0.10.2"
!pip install -q dm-haiku ml_collections absl-py biopython numpy

## 4. Clone this repository and AlphaFold 2, recording both commits

- **`REPO_REF`** selects the version of `src/spike_tpu_forward_pass.py` to run. Set it to a commit hash to reproduce a specific run. It must include the `--num_steady_state_runs` and `--profile_first_predict` flags; the cell checks this.
- **`ALPHAFOLD_REF = None`** uses AlphaFold 2's default branch, as the original runs did. Set a commit hash to pin it.

In [ ]:
import re, shutil, subprocess

REPO_URL = "https://github.com/lorenzopazienza/alphafold-tpu-benchmark.git"
REPO_REF = "PASTE_COMMIT_HASH_HERE"   # explicit commit hash, not a branch name
ALPHAFOLD_URL = "https://github.com/google-deepmind/alphafold.git"
ALPHAFOLD_REF = None    # None = default branch, or a commit hash

assert REPO_REF != "PASTE_COMMIT_HASH_HERE", "Set REPO_REF to the commit hash to run."
assert re.fullmatch(r"[0-9a-f]{7,40}", REPO_REF), (
    f"REPO_REF={REPO_REF!r} is not a commit hash (7-40 lowercase hex characters).")

def git(*args, cwd=None):
    return subprocess.run(["git", *args], cwd=cwd, check=True,
                          capture_output=True, text=True).stdout.strip()

for path in ("/content/bench", "/content/alphafold"):
    shutil.rmtree(path, ignore_errors=True)
git("clone", "-q", REPO_URL, "/content/bench")
git("checkout", "-q", REPO_REF, cwd="/content/bench")
git("clone", "-q", "--depth", "1", ALPHAFOLD_URL, "/content/alphafold")
if ALPHAFOLD_REF:
    git("fetch", "-q", "--depth", "1", "origin", ALPHAFOLD_REF, cwd="/content/alphafold")
    git("checkout", "-q", ALPHAFOLD_REF, cwd="/content/alphafold")

SCRIPT = "/content/bench/src/spike_tpu_forward_pass.py"
missing = [f for f in ("num_steady_state_runs", "profile_first_predict")
           if f'"{f}"' not in open(SCRIPT).read()]
assert not missing, f"REPO_REF={REPO_REF!r} predates the flags {missing}; use a newer ref."

provenance = {
    "repo_url": REPO_URL,
    "repo_ref": REPO_REF,
    "repo_commit": git("rev-parse", "HEAD", cwd="/content/bench"),
    "alphafold_url": ALPHAFOLD_URL,
    "alphafold_ref": ALPHAFOLD_REF,
    "alphafold_commit": git("rev-parse", "HEAD", cwd="/content/alphafold"),
}
print(json.dumps(provenance, indent=2))

## 5. Run the committed script: 5 steady-state calls, profiler off

- **Configuration flags:** `--model_name=model_3 --num_residues=118 --num_recycle=0 --precision=float32`. At 118 residues the script uses its fixed toy sequence (`TOY_SEQUENCE_118`).
- **Timing flags:** `--noprofile_first_predict --num_steady_state_runs=5`.
- **How it runs:** AlphaFold 2 is put on `PYTHONPATH` instead of copying the script into its checkout. The full log is saved next to the result JSON.

In [ ]:
import json, os, sys

RUN_TAG = "gpu-t4-repro"
RESULTS_DIR = "/content/results"
NUM_STEADY_STATE_RUNS = 5
COMMON_FLAGS = "--model_name=model_3 --num_residues=118 --num_recycle=0 --precision=float32"
PY = sys.executable

os.makedirs(RESULTS_DIR, exist_ok=True)
!PYTHONPATH=/content/alphafold {PY} {SCRIPT} --run_tag={RUN_TAG} {COMMON_FLAGS} --noprofile_first_predict --num_steady_state_runs={NUM_STEADY_STATE_RUNS} --results_dir={RESULTS_DIR} 2>&1 | tee {RESULTS_DIR}/log_{RUN_TAG}_noprofile.txt

result_path = f"{RESULTS_DIR}/result_{RUN_TAG}_model_3_len118_recycle0_float32_noprofile.json"
assert os.path.exists(result_path), f"No result JSON at {result_path}: the run failed, see the log above."
result = json.load(open(result_path))
assert result["profile_first_predict"] is False
assert result["num_steady_state_runs"] == NUM_STEADY_STATE_RUNS
assert len(result["steady_state_runs_seconds"]) == NUM_STEADY_STATE_RUNS
print(json.dumps({k: result[k] for k in (
    "init_params_seconds", "first_predict_compile_and_run_seconds",
    "steady_state_runs_seconds", "steady_state_mean_seconds",
    "steady_state_stdev_seconds")}, indent=2))

## 6. Optional: measure the profiler overhead on this runtime (discrepancy 11)

This runs the same configuration with `--profile_first_predict` and one steady-state call, in a separate process, since a process can only compile once.

The difference in `first_predict_compile_and_run_seconds` from step 5 estimates the cost of finalising the profiler trace, which inflated the original baseline. It is one run against one run, so it also contains run-to-run noise. The cost is one more compile; skip this cell if you only need the baseline.

In [ ]:
!PYTHONPATH=/content/alphafold {PY} {SCRIPT} --run_tag={RUN_TAG} {COMMON_FLAGS} --profile_first_predict --num_steady_state_runs=1 --results_dir={RESULTS_DIR} 2>&1 | tee {RESULTS_DIR}/log_{RUN_TAG}_profile.txt

profiled_path = f"{RESULTS_DIR}/result_{RUN_TAG}_model_3_len118_recycle0_float32.json"
assert os.path.exists(profiled_path), f"No result JSON at {profiled_path}: the run failed, see the log above."
profiled = json.load(open(profiled_path))
assert profiled["profile_first_predict"] is True
with_prof = profiled["first_predict_compile_and_run_seconds"]
without_prof = result["first_predict_compile_and_run_seconds"]
print(f"First predict, profiler on  (this step): {with_prof:.2f} s")
print(f"First predict, profiler off (step 5):    {without_prof:.2f} s")
print(f"Difference (one run each, includes noise): {with_prof - without_prof:.2f} s")

## 7. Save provenance and download everything

This step writes `environment_<RUN_TAG>.json` and `pip_freeze_<RUN_TAG>.txt` into the results directory, then downloads the whole directory as a zip.

- **`environment_<RUN_TAG>.json`:** commits, hardware, key package versions and flags.
- **`pip_freeze_<RUN_TAG>.txt`:** the full list of installed packages.
- **The zip:** result JSONs, logs, and the profiler trace if step 6 ran.

Copy the result and environment files into `results/` in the repo.

In [ ]:
import datetime, importlib.metadata, shutil, subprocess

PACKAGES = ["jax", "jaxlib", "jax-cuda12-plugin", "jax-cuda12-pjrt", "dm-haiku", "numpy", "tensorflow",
            "ml-collections", "absl-py", "biopython"]
versions = {}
for name in PACKAGES:
    try:
        versions[name] = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        versions[name] = None

environment = {
    "run_tag": RUN_TAG,
    "recorded_at_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    **provenance,
    "hardware": hardware,
    "python_executable": PY,
    "package_versions": versions,
    "flags": {"common": COMMON_FLAGS, "num_steady_state_runs": NUM_STEADY_STATE_RUNS},
    "result_files": sorted(f for f in os.listdir(RESULTS_DIR) if f.startswith("result_")),
}
with open(f"{RESULTS_DIR}/environment_{RUN_TAG}.json", "w") as f:
    json.dump(environment, f, indent=2)
with open(f"{RESULTS_DIR}/pip_freeze_{RUN_TAG}.txt", "w") as f:
    f.write(subprocess.run([PY, "-m", "pip", "freeze"], capture_output=True, text=True).stdout)
print(json.dumps({k: v for k, v in environment.items() if k != "hardware"}, indent=2))

archive = shutil.make_archive(f"/content/{RUN_TAG}_results", "zip", RESULTS_DIR)
from google.colab import files
files.download(archive)